# HADIS — Multi-Modal Transformer Fusion Training

**High Altitude Drone Intelligence System**  
Author: Prakash Tiwari | Chandigarh Engineering College (IKGPTU)

This notebook trains the Transformer Fusion model that combines:
- **Visual features** from frozen YOLOv8 backbone (512-d)
- **RF features** from frozen LSTM/CNN model (256-d)
- **Atmospheric features** from ISA data via AtmosphericMLP (32-d)

**Prerequisites:** YOLOv8 and LSTM/CNN weights must already be saved.

Loss: `CrossEntropyLoss + 0.3 * MSELoss(threat_score)`

---

In [ ]:
# Cell 2 — Install dependencies
!pip install -q ultralytics torch torchvision scikit-learn matplotlib seaborn pandas

In [ ]:
# Cell 3 — Mount Google Drive and clone HADIS repo
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/HADIS'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Tiwari1782/HADIS.git {REPO_DIR}
    print(f'[HADIS] Repository cloned to {REPO_DIR}')
else:
    print(f'[HADIS] Repository already exists at {REPO_DIR}')

In [ ]:
# Cell 4 — Imports and configuration
import sys
import os
import time
import glob

sys.path.append('/content/HADIS')
from config import PATHS, HYPERPARAMS, DRONE_CLASSES, THREAT_LEVELS, NUM_CLASSES

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from ultralytics import YOLO

# Import HADIS models
sys.path.append(os.path.join('/content/HADIS', 'hadis-ml'))
from lstm_cnn.model import get_model as get_rf_model, HADISRFClassifier
from transformer_fusion.fusion_model import (
    HADISFusionTransformer, AtmosphericMLP, get_fusion_model
)

print(f'[HADIS] Config loaded')
print(f'[HADIS] Transformer config: heads={HYPERPARAMS["transformer_heads"]}, '
      f'layers={HYPERPARAMS["transformer_layers"]}, epochs={HYPERPARAMS["transformer_epochs"]}')

In [ ]:
# Cell 5 — Verify GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f'[HADIS] GPU available: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('[HADIS] WARNING: No GPU detected.')

print(f'[HADIS] Using device: {device}')

In [ ]:
# Cell 6 — Load frozen YOLOv8 feature extractor
yolo_weights = os.path.join(PATHS['weights_yolo'], 'hadis_yolov8_best.pt')

if not os.path.exists(yolo_weights):
    raise FileNotFoundError(
        f'YOLOv8 weights not found at {yolo_weights}. '
        'Run train_yolov8.ipynb first.'
    )

yolo_model = YOLO(yolo_weights)
print(f'[HADIS] Loaded YOLOv8 weights from: {yolo_weights}')

# Access the PyTorch backbone for feature extraction
yolo_backbone = yolo_model.model.model

# Freeze all YOLOv8 parameters
for param in yolo_backbone.parameters():
    param.requires_grad = False
yolo_backbone.eval()

print('[HADIS] YOLOv8 backbone loaded and frozen.')

In [ ]:
# Cell 7 — Load frozen LSTM/CNN feature extractor
lstm_weights = os.path.join(PATHS['weights_lstm'], 'hadis_lstm_cnn_best.pt')

if not os.path.exists(lstm_weights):
    raise FileNotFoundError(
        f'LSTM/CNN weights not found at {lstm_weights}. '
        'Run train_lstm_cnn.ipynb first.'
    )

rf_model = get_rf_model(num_classes=NUM_CLASSES, seq_len=HYPERPARAMS['lstm_seq_len'])
rf_model.load_state_dict(torch.load(lstm_weights, map_location=device))
rf_model = rf_model.to(device)

# Freeze all LSTM/CNN parameters
for param in rf_model.parameters():
    param.requires_grad = False
rf_model.eval()

print(f'[HADIS] LSTM/CNN model loaded and frozen from: {lstm_weights}')

In [ ]:
# Cell 8 — Load ISA atmospheric data
isa_csv_path = PATHS['isa_data']

try:
    isa_df = pd.read_csv(isa_csv_path)
    print(f'[HADIS] ISA data loaded from: {isa_csv_path}')
    print(f'[HADIS] ISA shape: {isa_df.shape}')
    print(f'[HADIS] ISA columns: {list(isa_df.columns)}')
    print(isa_df.head())
except Exception as e:
    raise FileNotFoundError(f'ISA data not found at {isa_csv_path}: {e}')

# Normalise ISA data for consistent input
isa_columns = ['altitude_m', 'pressure_hPa', 'temperature_K', 'density', 'speed_of_sound']
isa_values = isa_df[isa_columns].values.astype(np.float32)
isa_mean = isa_values.mean(axis=0)
isa_std = isa_values.std(axis=0) + 1e-8
isa_values_normed = (isa_values - isa_mean) / isa_std

print(f'[HADIS] ISA normalised. Rows available: {len(isa_values_normed)}')

In [ ]:
# Cell 9 — FusionDataset class

class FusionDataset(Dataset):
    """Dataset for multi-modal fusion training.

    For each sample, extracts:
    - yolo_feat: Visual features through frozen YOLOv8 backbone
    - rf_feat: RF features through frozen LSTM/CNN (minus classification head)
    - atmos_vec: Random ISA row for atmospheric conditions
    - class_label: Drone class index
    - threat_level: Normalised threat score

    Since we need to process images through YOLOv8 and RF signals through
    LSTM/CNN, this dataset pre-computes features at construction time to
    avoid running inference during training.
    """

    def __init__(self, image_dir, rf_dir, isa_values_normed, yolo_backbone,
                 rf_model, device, seq_len=256, max_samples_per_class=2000):
        """Initialise the fusion dataset.

        Args:
            image_dir: Path to drone image directory (YOLO format).
            rf_dir: Path to DroneRF data directory.
            isa_values_normed: Normalised ISA data array.
            yolo_backbone: Frozen YOLOv8 backbone model.
            rf_model: Frozen LSTM/CNN model.
            device: Torch device.
            seq_len: RF signal window length.
            max_samples_per_class: Cap samples per class for balance.
        """
        self.isa_values = isa_values_normed
        self.samples = []  # list of (yolo_feat, rf_feat, class_idx, threat_level)

        print('[HADIS] Building FusionDataset...')

        # --- Extract visual features from drone images ---
        yolo_feats_by_class = {}
        img_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']

        # Scan train images directory
        train_img_dir = os.path.join(image_dir, 'images', 'train')
        if os.path.exists(train_img_dir):
            image_files = []
            for ext in img_extensions:
                image_files.extend(glob.glob(os.path.join(train_img_dir, ext)))

            print(f'[HADIS]   Found {len(image_files)} training images')

            # Extract features in batches through YOLOv8
            from torchvision import transforms
            from PIL import Image

            preprocess = transforms.Compose([
                transforms.Resize((640, 640)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225]),
            ])

            visual_features = []
            with torch.no_grad():
                for img_path in image_files[:max_samples_per_class * NUM_CLASSES]:
                    try:
                        img = Image.open(img_path).convert('RGB')
                        img_tensor = preprocess(img).unsqueeze(0).to(device)

                        # Forward through YOLOv8 backbone layers
                        x = img_tensor
                        for i, layer in enumerate(yolo_backbone):
                            x = layer(x)
                            if i == 9:  # Stop after backbone + neck
                                break

                        # Global average pool to get a feature vector
                        feat = torch.nn.functional.adaptive_avg_pool2d(x, 1)
                        feat = feat.flatten(1)  # (1, C)

                        # Pad or truncate to 512
                        if feat.shape[1] < 512:
                            feat = torch.nn.functional.pad(feat, (0, 512 - feat.shape[1]))
                        else:
                            feat = feat[:, :512]

                        visual_features.append(feat.cpu())
                    except Exception as e:
                        continue

            print(f'[HADIS]   Extracted {len(visual_features)} visual features')
        else:
            print(f'[HADIS] WARNING: Train image directory not found: {train_img_dir}')
            visual_features = []

        # --- Extract RF features ---
        rf_features = []
        rf_labels = []
        FOLDER_CLASS_MAP = {
            'ar': 0, 'bebop': 0, 'bepop': 0, 'phantom': 0, 'background': 5,
        }

        if os.path.exists(rf_dir):
            for subfolder in sorted(os.listdir(rf_dir)):
                subfolder_path = os.path.join(rf_dir, subfolder)
                if not os.path.isdir(subfolder_path):
                    continue

                label = None
                folder_lower = subfolder.lower()
                for keyword, cls_idx in FOLDER_CLASS_MAP.items():
                    if keyword in folder_lower:
                        label = cls_idx
                        break
                if label is None:
                    continue

                data_files = []
                for ext in ['*.csv', '*.npy']:
                    data_files.extend(glob.glob(os.path.join(subfolder_path, '**', ext), recursive=True))

                for fpath in data_files:
                    try:
                        if fpath.endswith('.csv'):
                            signal = pd.read_csv(fpath, header=None).values.flatten().astype(np.float32)
                        else:
                            signal = np.load(fpath).flatten().astype(np.float32)

                        n_windows = len(signal) // seq_len
                        for i in range(min(n_windows, max_samples_per_class)):
                            window = signal[i * seq_len: (i + 1) * seq_len]
                            mean, std = window.mean(), window.std()
                            if std > 1e-8:
                                window = (window - mean) / std
                            else:
                                window = window - mean

                            tensor = torch.tensor(window, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1,1,seq_len)
                            with torch.no_grad():
                                feat = rf_model.extract_features(tensor.to(device))  # (1, 256)
                            rf_features.append(feat.cpu())
                            rf_labels.append(label)
                    except Exception:
                        continue

            print(f'[HADIS]   Extracted {len(rf_features)} RF features')
        else:
            print(f'[HADIS] WARNING: DroneRF directory not found: {rf_dir}')

        # --- Pair visual and RF features ---
        n_samples = min(len(visual_features), len(rf_features))
        if n_samples == 0:
            print('[HADIS] WARNING: No paired samples available. Check data paths.')
            return

        for i in range(n_samples):
            label = rf_labels[i]
            class_name = DRONE_CLASSES[label]
            threat = THREAT_LEVELS.get(class_name, 3) / 4.0  # Normalise to [0, 1]

            self.samples.append((
                visual_features[i].squeeze(0),   # (512,)
                rf_features[i].squeeze(0),        # (256,)
                label,
                threat,
            ))

        print(f'[HADIS] FusionDataset built: {len(self.samples)} paired samples')

    def __len__(self):
        """Return the number of samples."""
        return len(self.samples)

    def __getitem__(self, idx):
        """Return (yolo_feat, rf_feat, atmos_vec, class_label, threat_level)."""
        yolo_feat, rf_feat, label, threat = self.samples[idx]

        # Sample a random ISA row for atmospheric conditions
        isa_idx = np.random.randint(0, len(self.isa_values))
        atmos_vec = torch.tensor(self.isa_values[isa_idx], dtype=torch.float32)

        return yolo_feat, rf_feat, atmos_vec, label, threat


# Build dataset
fusion_dataset = FusionDataset(
    image_dir=PATHS['drone_dataset'],
    rf_dir=PATHS['dronerf'],
    isa_values_normed=isa_values_normed,
    yolo_backbone=yolo_backbone,
    rf_model=rf_model,
    device=device,
    seq_len=HYPERPARAMS['lstm_seq_len'],
)

print(f'[HADIS] FusionDataset ready: {len(fusion_dataset)} samples')

In [ ]:
# Cell 10 — Train/Val split and DataLoaders
total = len(fusion_dataset)
train_size = int(0.85 * total)
val_size = total - train_size

generator = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(fusion_dataset, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_ds, batch_size=HYPERPARAMS['lstm_batch'], shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=HYPERPARAMS['lstm_batch'], shuffle=False,
                        num_workers=2, pin_memory=True)

print(f'[HADIS] Split: train={train_size}, val={val_size}')

In [ ]:
# Cell 11 — Create fusion model and resume logic
fusion_model, atmos_mlp = get_fusion_model(num_classes=NUM_CLASSES)
fusion_model = fusion_model.to(device)
atmos_mlp = atmos_mlp.to(device)

# Only train fusion model and atmospheric MLP (YOLOv8 and LSTM/CNN are frozen)
trainable_params = list(fusion_model.parameters()) + list(atmos_mlp.parameters())
optimizer = optim.Adam(trainable_params, lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=HYPERPARAMS['transformer_epochs'])

cls_criterion = nn.CrossEntropyLoss()
reg_criterion = nn.MSELoss()
threat_weight = 0.3

weights_dir = PATHS['weights_fusion']
os.makedirs(weights_dir, exist_ok=True)

checkpoint_path = os.path.join(weights_dir, 'checkpoint_last.pt')
start_epoch = 0
best_val_loss = float('inf')
train_losses, val_losses = [], []

if os.path.exists(checkpoint_path):
    try:
        ckpt = torch.load(checkpoint_path, map_location=device)
        fusion_model.load_state_dict(ckpt['fusion_state_dict'])
        atmos_mlp.load_state_dict(ckpt['atmos_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        train_losses = ckpt.get('train_losses', [])
        val_losses = ckpt.get('val_losses', [])
        print(f'[HADIS] Resumed from epoch {start_epoch} | Best val loss: {best_val_loss:.4f}')
    except Exception as e:
        print(f'[HADIS] WARNING: Failed to load checkpoint: {e}')
        start_epoch = 0
else:
    print('[HADIS] No checkpoint found. Starting fresh training.')

fusion_params = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
atmos_params = sum(p.numel() for p in atmos_mlp.parameters() if p.requires_grad)
print(f'[HADIS] Trainable params: fusion={fusion_params:,}, atmos={atmos_params:,}')

In [ ]:
# Cell 12 — Training loop
num_epochs = HYPERPARAMS['transformer_epochs']

for epoch in range(start_epoch, num_epochs):
    epoch_start = time.time()

    # --- Training phase ---
    fusion_model.train()
    atmos_mlp.train()
    running_loss = 0.0
    running_cls_loss = 0.0
    running_reg_loss = 0.0
    correct = 0
    total_samples = 0

    for yolo_feat, rf_feat, atmos_vec, labels, threats in train_loader:
        yolo_feat = yolo_feat.to(device)
        rf_feat = rf_feat.to(device)
        atmos_vec = atmos_vec.to(device)
        labels = labels.to(device)
        threats = threats.float().to(device)

        optimizer.zero_grad()

        # Forward through atmospheric MLP
        atmos_feat = atmos_mlp(atmos_vec)  # (batch, 32)

        # Forward through fusion transformer
        class_logits, threat_score = fusion_model(yolo_feat, rf_feat, atmos_feat)

        # Combined loss
        cls_loss = cls_criterion(class_logits, labels)
        reg_loss = reg_criterion(threat_score.squeeze(-1), threats)
        loss = cls_loss + threat_weight * reg_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()

        bs = yolo_feat.size(0)
        running_loss += loss.item() * bs
        running_cls_loss += cls_loss.item() * bs
        running_reg_loss += reg_loss.item() * bs
        _, predicted = class_logits.max(1)
        correct += predicted.eq(labels).sum().item()
        total_samples += bs

    train_loss = running_loss / total_samples
    train_acc = correct / total_samples

    # --- Validation phase ---
    fusion_model.eval()
    atmos_mlp.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for yolo_feat, rf_feat, atmos_vec, labels, threats in val_loader:
            yolo_feat = yolo_feat.to(device)
            rf_feat = rf_feat.to(device)
            atmos_vec = atmos_vec.to(device)
            labels = labels.to(device)
            threats = threats.float().to(device)

            atmos_feat = atmos_mlp(atmos_vec)
            class_logits, threat_score = fusion_model(yolo_feat, rf_feat, atmos_feat)

            cls_loss = cls_criterion(class_logits, labels)
            reg_loss = reg_criterion(threat_score.squeeze(-1), threats)
            loss = cls_loss + threat_weight * reg_loss

            bs = yolo_feat.size(0)
            val_running_loss += loss.item() * bs
            _, predicted = class_logits.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += bs

    val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total

    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    elapsed = time.time() - epoch_start
    lr = optimizer.param_groups[0]['lr']
    print(f'[HADIS] Epoch {epoch+1}/{num_epochs} | '
          f'Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | '
          f'LR: {lr:.6f} | Time: {elapsed:.1f}s')

    # --- Save checkpoint every epoch ---
    try:
        ckpt_data = {
            'epoch': epoch,
            'fusion_state_dict': fusion_model.state_dict(),
            'atmos_state_dict': atmos_mlp.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
        }
        torch.save(ckpt_data, checkpoint_path)
        print(f'[HADIS] Checkpoint saved: {checkpoint_path}')
    except Exception as e:
        print(f'[HADIS] WARNING: Failed to save checkpoint: {e}')

    # --- Save best model ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_path = os.path.join(weights_dir, 'hadis_fusion_best.pt')
        try:
            torch.save({
                'fusion_state_dict': fusion_model.state_dict(),
                'atmos_state_dict': atmos_mlp.state_dict(),
            }, best_path)
            print(f'[HADIS] New best model saved: {best_path} (loss={best_val_loss:.4f})')
        except Exception as e:
            print(f'[HADIS] WARNING: Failed to save best model: {e}')

print(f'[HADIS] Training complete. Best validation loss: {best_val_loss:.4f}')

In [ ]:
# Cell 13 — Plot training curves
logs_dir = PATHS['logs']
os.makedirs(logs_dir, exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train Loss', linewidth=2)
ax.plot(val_losses, label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (CE + 0.3*MSE)')
ax.set_title('HADIS Transformer Fusion - Training Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

try:
    plot_path = os.path.join(logs_dir, 'transformer_fusion_training_curves.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f'[HADIS] Training curves saved to: {plot_path}')
except Exception as e:
    print(f'[HADIS] WARNING: Failed to save plot: {e}')

plt.show()
print(f'[HADIS] Final weights: {os.path.join(weights_dir, "hadis_fusion_best.pt")}')